# Docker: Part 2
## Volumes, Networks & Docker Compose

**Topics covered:**
1. The Problem: Containers Forget Everything
2. Volumes (named volumes vs bind mounts)
3. Networks: How Containers Find Each Other
4. Docker Compose: One File, Whole System
5. Health Checks & Startup Order
6. Scaling an Image into Many Containers
7. Hardening: `.dockerignore` & Non-Root User
8. Cleanup & Cheat Sheet

---
> **Context:** In Part 1 you ran one container. Real systems are an API + a database + a cache. Part 2 is how those pieces keep their data and talk to each other. This is the system design diagram, made real.

## 1. The Problem: Containers Forget Everything

A container's filesystem is a thin writable layer on top of the read-only image. **Delete the container and that layer is gone.**

**Analogy: a hotel room.** You can rearrange the furniture, but when you check out, housekeeping resets the room. Anything you want to keep must go in a **safe deposit box** outside the room. That box is a **volume**.

Run the cell below. We write a file inside a container, remove the container, then start a fresh one.

In [23]:
# Write a file inside a container, then throw the container away
!docker run --name tmp1 alpine sh -c 'echo "important data" > /note.txt && cat /note.txt'
!docker rm tmp1
# A brand new container from the same image: is the file still there?
!docker run --rm alpine cat /note.txt

important data
tmp1
cat: can't open '/note.txt': No such file or directory


The second run fails with *No such file*. This is why a database in a plain container loses all its rows when the container is replaced.

---
## 2. Volumes

Docker has two ways to keep data outside the container:

| | **Named volume** | **Bind mount** |
|---|---|---|
| Syntax | `-v pgdata:/var/lib/postgresql/data` | `-v $(pwd)/code:/app` |
| Who manages it | Docker (lives inside Docker's storage) | You (a real folder on your Mac) |
| Best for | Database files, anything that must persist | Live-editing code during development |
| Analogy | A storage unit the warehouse looks after | Your own garage you can walk into |

The left side of the colon is the **outside** (volume name or host path). The right side is the **path inside the container**.

In [24]:
# Named volume: data lives in "notes", not in the container
!docker volume create notes
!docker run --rm -v notes:/data alpine sh -c 'echo "important data" > /data/note.txt'
# A completely different container reads it back
!docker run --rm -v notes:/data alpine cat /data/note.txt

notes
important data


In [25]:
# Docker tracks volumes separately from containers
!docker volume ls --filter name=notes
!docker volume inspect notes --format '{{{{.Name}}}} -> {{{{.Driver}}}}'

DRIVER    VOLUME NAME
local     notes
notes -> local


### Bind mount: edit on your Mac, see it inside instantly
Great in development: you change code in VS Code and the container sees the change without a rebuild.

In [26]:
import os, pathlib
pathlib.Path("shared_demo").mkdir(exist_ok=True)
pathlib.Path("shared_demo/hello.txt").write_text("written on the Mac\n")

# Mount the Mac folder at /mnt inside the container and read the file
!docker run --rm -v "{os.getcwd()}/shared_demo":/mnt alpine cat /mnt/hello.txt

written on the Mac


In [27]:
# The container can write back too, and you see it on your Mac
!docker run --rm -v "{os.getcwd()}/shared_demo":/mnt alpine sh -c 'echo "written in the container" >> /mnt/hello.txt'
print(pathlib.Path("shared_demo/hello.txt").read_text())

written on the Mac
written in the container



**Rule of thumb:** database files -> named volume. Source code while developing -> bind mount.

**Exercise:** a Postgres container stores its data in `/var/lib/postgresql/data`. Write the `-v` flag that keeps it in a volume called `pgdata`.

# Docker Volumes: Persisting Postgres Data

```bash
docker run -d --name db -e POSTGRES_PASSWORD=secret -v pgdata:/var/lib/postgresql/data postgres:16
```

## Why it's written that way

`-v pgdata:/var/lib/postgresql/data` has two sides, split by the colon:

| Part | Meaning |
|---|---|
| Left of the colon: `pgdata` | The outside. The name of the volume. Docker creates it if it doesn't exist. |
| Right of the colon: `/var/lib/postgresql/data` | The path inside the container where Postgres writes its files. Anything written there goes into the volume. |

## What it gives you

You can run `docker rm -f db` and start a new Postgres container with the same `-v pgdata:...` flag. All your tables and rows are still there, because the data lived in the volume (the safe deposit box), not in the container.

```bash
docker rm -f db
docker run -d --name db -e POSTGRES_PASSWORD=secret -v pgdata:/var/lib/postgresql/data postgres:16
```

## Common mistake

The `-v` flag has two modes. Docker decides which one you get by looking at the **left side of the colon**.

| Left side | Type | Where the data lives |
|---|---|---|
| `pgdata` (no slashes) | Named volume | Inside Docker's own storage, managed by Docker |
| `/Users/you/pgdata` or `./pgdata` (has slashes) | Bind mount | A real folder on your computer |

**Named volume**

```bash
-v pgdata:/var/lib/postgresql/data
```

Docker creates and manages a storage area called `pgdata`. You don't choose where it lives; Docker keeps it in its own internal folders. This is what you want for databases.

**Bind mount**

```bash
-v /Users/you/pgdata:/var/lib/postgresql/data
```

Docker skips its own storage and links a real folder on your computer into the container. Whatever the container writes shows up in that folder, and you can open it in Finder.

**The mistake** is writing a slash path when you meant a named volume. For example, `-v ./pgdata:/var/lib/...` looks like "a volume called pgdata", but the slash tells Docker "this is a folder path", so you silently get a bind mount instead. For Postgres this can cause permission errors, slower performance on Mac, and a data folder sitting somewhere you might accidentally delete.

**Rule of thumb:** no slashes means named volume. Slashes mean bind mount.

---
## 3. Networks: How Containers Find Each Other

Your API container needs to reach a Redis container. Two problems:

1. Containers have their own network, so `localhost` inside the API means *the API itself*, not Redis.
2. Container IP addresses change every time a container restarts.

**Analogy: a phone contacts list.** You do not memorise phone numbers (IPs), you call "Mum" (a name). On a **user-defined network**, Docker runs a tiny DNS service so containers can call each other **by name**.

In [28]:
# Create a network and start a "server" container on it
!docker network create demo-net
!docker run -d --name cache --network demo-net redis:7-alpine

af9abe609c9e21a1ee2fa5d104835156c807f00899614b478318d78c35004baf
22c80e946fc78e8663fece8f55dd1200dcee35d6f4f0ff1f0f24ccf54ae8a784


In [29]:
# A second container on the same network reaches it BY NAME
!docker run --rm --network demo-net redis:7-alpine redis-cli -h cache ping

PONG


In [30]:
# A container NOT on that network cannot resolve the name
!docker run --rm redis:7-alpine redis-cli -h cache ping

Could not connect to Redis at cache:6379: Name does not resolve


The first prints `PONG`. The second fails with a name resolution error. Networks are also **isolation**: only containers you put on the same network can talk.

> Note: you did **not** publish a port with `-p` for `cache`. Container-to-container traffic on a shared network doesn't need it. `-p` is only for traffic from **outside** (your browser, your Mac).

In [31]:
!docker rm -f cache
!docker network rm demo-net

cache
demo-net


---
## 4. Docker Compose: One File, Whole System

So far you typed long commands with many flags: a network, a volume, ports, env vars, start order. Forgetting one flag breaks things. **Compose** puts all of it in one `docker-compose.yml` that you start with a single command.

**Analogy:** instead of telling a moving crew each instruction by phone, you hand them a **written floor plan** and say "set it up".

Open `../hello_compose/docker-compose.yml`:

```yaml
services:
  api:
    build: .                      # build image from the Dockerfile here
    ports: ["8000:8000"]          # same as -p 8000:8000
    environment:
      REDIS_HOST: redis           # same as -e REDIS_HOST=redis
    depends_on:
      redis:
        condition: service_healthy

  redis:
    image: redis:7-alpine         # use a ready-made image
    command: redis-server --appendonly yes
    volumes:
      - redisdata:/data           # same as -v redisdata:/data
    healthcheck: ...

volumes:
  redisdata:                      # declares the named volume
```

Each key under `services:` is a **service**. Compose automatically:
- creates a **network** for the project,
- makes each service name (`api`, `redis`) a **hostname** on it,
- creates the declared **volumes**.

That's why `main.py` can say `redis.Redis(host="redis")` and it just works.

In [32]:
# Start both services: build the api image, pull redis, create network + volume
!docker compose -f ../hello_compose/docker-compose.yml up -d --build

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (1/1)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 673B                                             0.0s
[+] Building 0.2s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 673B                                             0.0s
[+] Building 0.3s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 673B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 373B

In [33]:
!docker compose -f ../hello_compose/docker-compose.yml ps

NAME                    IMAGE               COMMAND                  SERVICE   CREATED         STATUS                   PORTS
hello_compose-api-1     hello_compose-api   "uvicorn main:app --…"   api       4 seconds ago   Up Less than a second    0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
hello_compose-redis-1   redis:7-alpine      "docker-entrypoint.s…"   redis     4 seconds ago   Up 4 seconds (healthy)   6379/tcp


In [34]:
# Hit the API three times. The counter lives in the redis container.
!sleep 2
!curl -s localhost:8000/hits
!curl -s localhost:8000/hits
!curl -s localhost:8000/hits

{"hits":1,"served_by":"2ef5c61efa03"}{"hits":2,"served_by":"2ef5c61efa03"}{"hits":3,"served_by":"2ef5c61efa03"}

### Prove the volume works
`docker compose down` **destroys the containers and network** but keeps the named volume. Bring it back up and the counter should continue from 3, not reset to 1.

In [35]:
!docker compose -f ../hello_compose/docker-compose.yml down
!docker compose -f ../hello_compose/docker-compose.yml up -d

[+] Running 0/1
 ⠋ Container hello_compose-api-1  Stopp...                                 0.1s 
[+] Running 0/1
 ⠙ Container hello_compose-api-1  Stopp...                                 0.2s 
[+] Running 0/1
 ⠹ Container hello_compose-api-1  Stopp...                                 0.3s 
[+] Running 0/1
 ⠸ Container hello_compose-api-1  Stopp...                                 0.4s 
[+] Running 0/1
 ⠼ Container hello_compose-api-1  Remov...                                 0.5s 
[+] Running 1/2
 ✔ Container hello_compose-api-1    Rem...                                 0.5s 
 ⠋ Container hello_compose-redis-1  S...                                   0.1s 
[+] Running 1/2
 ✔ Container hello_compose-api-1    Rem...                                 0.5s 
 ⠙ Container hello_compose-redis-1  R...                                   0.2s 
[+] Running 2/3
 ✔ Container hello_compose-api-1    Rem...                                 0.5s 
 ✔ Container hello_compose-redis-1  R...                      

In [36]:
!sleep 3
!curl -s localhost:8000/hits

{"hits":4,"served_by":"ff82056ee7a8"}

You should see `4`. Brand-new containers, same data. That's the safe deposit box from Section 1.

| Command | Containers | Network | Volumes |
|---|---|---|---|
| `docker compose stop` | stopped, kept | kept | kept |
| `docker compose down` | **removed** | **removed** | kept |
| `docker compose down -v` | removed | removed | **removed (data lost)** |

---
## 5. Health Checks & Startup Order

`depends_on` alone only waits for the container to **start**, not to be **ready**. Redis may take a moment before it accepts connections, and an API that boots first would crash on its first request.

A **healthcheck** is a command Docker runs repeatedly to ask "are you actually working?":

```yaml
healthcheck:
  test: ["CMD", "redis-cli", "ping"]   # healthy if this exits 0
  interval: 3s
  retries: 5
```

and `condition: service_healthy` makes the API wait for it. **Analogy:** a restaurant opening. "The chef arrived" (started) is not the same as "the kitchen is hot and ready" (healthy).

In [37]:
# STATUS shows "(healthy)" once the check passes
!docker compose -f ../hello_compose/docker-compose.yml ps --format 'table {{{{.Service}}}}\t{{{{.Status}}}}'

SERVICE   STATUS
api       Up 4 seconds
redis     Up 7 seconds (healthy)


---
## 6. Scaling an Image into Many Containers

This is the horizontal scaling from system design. The compose file publishes a **port range** (`8000-8002:8000`), so each copy of the API grabs the next free host port.

In [38]:
# Run 3 copies of the api service
!docker compose -f ../hello_compose/docker-compose.yml up -d --scale api=3
!sleep 3
!docker compose -f ../hello_compose/docker-compose.yml ps --format 'table {{{{.Service}}}}\t{{{{.Ports}}}}'

[+] Running 3/4
 ✔ Container hello_compose-redis-1  R...                                   0.0s 
 ✔ Container hello_compose-api-1    Run...                                 0.0s 
 ⠋ Container hello_compose-api-3    Cre...                                 0.0s 
 ✔ Container hello_compose-api-2    Cre...                                 0.0s 
[+] Running 3/4
 ⠙ Container hello_compose-redis-1  W...                                   0.2s 
 ✔ Container hello_compose-api-1    Run...                                 0.0s 
 ✔ Container hello_compose-api-3    Cre...                                 0.1s 
 ✔ Container hello_compose-api-2    Cre...                                 0.0s 
[+] Running 3/4
 ⠹ Container hello_compose-redis-1  W...                                   0.3s 
 ✔ Container hello_compose-api-1    Run...                                 0.0s 
 ✔ Container hello_compose-api-3    Cre...                                 0.1s 
 ✔ Container hello_compose-api-2    Cre...                   

In [39]:
# Three different containers (different served_by), ONE shared counter
!curl -s localhost:8000/hits
!curl -s localhost:8001/hits
!curl -s localhost:8002/hits

{"hits":5,"served_by":"ff82056ee7a8"}

Each copy reports a different `served_by` container ID, but the `hits` number keeps climbing across all of them, because the *state* lives in Redis, not inside the API containers. **Stateless app + shared state store** is the pattern that makes scaling possible.

Real systems put a **load balancer** (nginx, a cloud LB) in front and publish only *its* port, so users see one address.

**Exercise:** why is `INCR` in Redis better for the counter than the API keeping `hits = 0` in a Python variable?

**Answer:** a Python variable lives inside one container, so it's private, temporary and can't be shared. Redis `INCR` lives outside the API, so it's shared, durable and safe under concurrency.

**1. Each container would have its own counter.**
With three API containers, each has its own `hits = 0`. You'd see 1, 1, 1, then 2, 2, 2, depending on which copy a request lands on. There is no single true total. With Redis, all copies update the same number, which is why the count kept climbing across different containers.

**2. The variable dies with the container.**
Restart or replace the container (a deploy, a crash, `docker compose down`) and `hits` goes back to 0. Redis keeps its data in the `redisdata` volume, so the count survived `down` and `up`.

**3. `INCR` is atomic.**
`hits = hits + 1` is really three steps: read, add, write. If two requests arrive together:

`Request A reads 10      Request B reads 10
Request A writes 11     Request B writes 11    <- should be 12, one hit lost`

Redis runs `INCR` as a single indivisible operation, so two simultaneous calls always give 11 then 12, never a lost update.

**Lesson:** keep API containers **stateless** and push anything that must be remembered into a shared store (Redis, Postgres). Then you can start, stop, replace and multiply API containers freely without losing data. That is what makes horizontal scaling work.

**Analogy:** three cashiers each keeping a tally in their head, which vanishes when they go home, versus one shared ledger on the wall that everyone writes to and that stays overnight. Redis is the ledger.




---
## 7. Hardening: `.dockerignore` & Non-Root User

Two small habits that separate hobby images from production ones.

### `.dockerignore`
Anything in the build context gets copied by `COPY . .`. Without a `.dockerignore`, your `.env` secrets, `.git` history and `__pycache__` end up **inside the image**, and anyone who can pull the image can read them. See `../hello_compose/.dockerignore`.

### Don't run as root
By default the process inside a container is **root**. If someone exploits your app, they'd be root inside the container. Two lines in the Dockerfile fix that:

```dockerfile
RUN useradd --create-home appuser
USER appuser
```

**Analogy:** give the cashier a till key, not the master key to the whole building.

In [40]:
# Compare: who is the process inside our containers?
!docker compose -f ../hello_compose/docker-compose.yml up -d
!docker compose -f ../hello_compose/docker-compose.yml exec api whoami
!docker run --rm alpine whoami

[+] Running 4/4
 ✔ Container hello_compose-redis-1  R...                                   0.0s 
 ✔ Container hello_compose-api-1    Run...                                 0.0s 
 ✔ Container hello_compose-api-3    Rem...                                 0.0s 
 ✔ Container hello_compose-api-2    Rem...                                 0.0s 
[+] Running 3/4
 ⠙ Container hello_compose-redis-1  W...                                   0.1s 
 ✔ Container hello_compose-api-1    Run...                                 0.0s 
 ✔ Container hello_compose-api-3    Rem...                                 0.0s 
 ✔ Container hello_compose-api-2    Rem...                                 0.0s 
[+] Running 3/4
 ⠹ Container hello_compose-redis-1  W...                                   0.2s 
 ✔ Container hello_compose-api-1    Run...                                 0.0s 
 ✔ Container hello_compose-api-3    Rem...                                 0.0s 
 ✔ Container hello_compose-api-2    Rem...                   

---
## 8. Cleanup

In [41]:
# Remove containers + network + volume of the compose project, plus the demo volume and folder
!docker compose -f ../hello_compose/docker-compose.yml down -v
!docker volume rm notes
import shutil; shutil.rmtree("shared_demo", ignore_errors=True)
!docker ps -a

[+] Running 0/1
 ⠋ Container hello_compose-api-1  Stopp...                                 0.1s 
[+] Running 0/1
 ⠙ Container hello_compose-api-1  Stopp...                                 0.2s 
[+] Running 0/1
 ⠹ Container hello_compose-api-1  Stopp...                                 0.3s 
[+] Running 0/1
 ⠸ Container hello_compose-api-1  Stopp...                                 0.4s 
[+] Running 1/2
 ✔ Container hello_compose-api-1    Rem...                                 0.5s 
 ⠋ Container hello_compose-redis-1  S...                                   0.0s 
[+] Running 1/2
 ✔ Container hello_compose-api-1    Rem...                                 0.5s 
 ⠙ Container hello_compose-redis-1  S...                                   0.1s 
[+] Running 3/4
 ✔ Container hello_compose-api-1    Rem...                                 0.5s 
 ✔ Container hello_compose-redis-1  R...                                   0.1s 
 ✔ Volume hello_compose_redisdata   Re...                                  0.0

In [42]:
# Optional: see how much disk Docker is using, and reclaim unused stuff
!docker system df
# !docker system prune        # uncomment to remove stopped containers, unused networks, dangling images

TYPE            TOTAL     ACTIVE    SIZE      RECLAIMABLE
Images          20        7         16.73GB   9.296GB (55%)
Containers      9         2         122.9kB   98.3kB (80%)
Local Volumes   17        6         446.4MB   114.7MB (25%)
Build Cache     61        0         10.48GB   3.27GB


# Part 2 Cheat Sheet

## Volumes
| Command | What it does |
|---|---|
| `docker volume create NAME` | Create a named volume |
| `docker volume ls` | List volumes |
| `docker volume inspect NAME` | Show details |
| `docker volume rm NAME` | Delete (data is gone) |
| `docker run -v NAME:/path IMAGE` | Named volume (persistent data) |
| `docker run -v "$(pwd)/dir":/path IMAGE` | Bind mount (live-edit code) |

## Networks
| Command | What it does |
|---|---|
| `docker network create NAME` | Create a user-defined network |
| `docker network ls` | List networks |
| `docker run --network NAME IMAGE` | Attach container (reachable by name) |
| `docker network rm NAME` | Delete a network |

## Compose
| Command | What it does |
|---|---|
| `docker compose up -d --build` | Build and start everything in background |
| `docker compose ps` | Status of services (incl. health) |
| `docker compose logs -f SERVICE` | Follow a service's logs |
| `docker compose exec SERVICE CMD` | Run a command inside a service |
| `docker compose stop` | Stop, keep everything |
| `docker compose down` | Remove containers + network, **keep volumes** |
| `docker compose down -v` | Also remove volumes (**data lost**) |


# Part 2 Summary: Volumes, Networks & Compose

## The big idea in one line

A real app is several containers (API, database, cache). Containers are disposable, so you need a way to keep data, a way for containers to find each other, and a way to start everything with one command.

| Problem | Solution | Analogy |
|---|---|---|
| Container deleted, data gone | Volume | Safe deposit box outside the hotel room |
| API can't find the database | Network + service name | Phone contacts: call "Mum", not a number |
| Too many flags to remember | Docker Compose | A written floor plan handed to the moving crew |
| App starts before DB is ready | Healthcheck + `depends_on` | "Chef arrived" isn't the same as "kitchen is hot" |
| Need more capacity | Scale a stateless service | Open more identical checkout counters |

## 1. Volumes: keeping data

A container's files vanish when it is removed. A volume lives outside it.

```bash
# Without a volume: data lost when the container is removed
docker run --name db postgres:16          # rows die with the container

# With a named volume: data survives
docker run -v pgdata:/var/lib/postgresql/data postgres:16
#              ^outside    ^inside the container
```

- **Named volume** (`pgdata:/path`): Docker manages it. Use for databases.
- **Bind mount** (`/Users/you/code:/app`): a real folder on your Mac. Use for live-editing code in development.
- **Quick test:** no slash on the left means named volume. A path on the left means bind mount.

## 2. Networks: containers finding each other

- `localhost` inside a container means that container itself, not its neighbour.
- IP addresses change on every restart, so never rely on them.
- On a shared network, Docker acts as a phone book, and each container is reachable by name.

```python
redis.Redis(host="redis")     # "redis" is the other container's name
```

- `-p 8000:8000` is only for traffic from outside (your browser).
- Container-to-container traffic on the same network needs no `-p`.

## 3. Docker Compose: the whole system in one file

```bash
docker compose up -d --build    # build + start everything
docker compose down             # remove containers + network, KEEP data
docker compose down -v          # also delete volumes (data lost!)
```

Compose automatically creates one network for the project, makes every service name a hostname, and creates the declared volumes.

## 4. The docker-compose.yml, line by line

```yaml
services:                          # (A) everything that runs
  api:                             # (B) service name, also its hostname
    build: .                       # (C) build from the Dockerfile in this folder
    ports:
      - "8000:8000"                # (D) HOST_PORT:CONTAINER_PORT
    environment:
      REDIS_HOST: redis            # (E) config passed to the app
    depends_on:
      redis:
        condition: service_healthy # (F) wait until redis is READY

  redis:                           # (G) second service
    image: redis:7-alpine          # (H) ready-made image, no build needed
    command: redis-server --appendonly yes   # (I) override the start command
    volumes:
      - redisdata:/data            # (J) mount named volume at /data
    healthcheck:                   # (K) how Docker tests "is it working?"
      test: ["CMD", "redis-cli", "ping"]
      interval: 3s
      timeout: 3s
      retries: 5

volumes:                           # (L) declare the named volume
  redisdata:
```

| Part | What it is | Why it's needed |
|---|---|---|
| A `services:` | Top-level list of containers | The core of the file: one entry per container |
| B `api:` name | Service name | Becomes the hostname other containers use, which is why `host="redis"` works |
| C `build: .` | Build the image from a Dockerfile | Use `build` for your own code |
| H `image:` | Pull a ready image | Use `image` for third-party software (redis, postgres, nginx) |
| D `ports` | Publish to your Mac | Without it, your browser can't reach the API. Format is `HOST:CONTAINER` |
| E `environment` | Env vars for the app | Keeps config out of the image. Same image, different settings per environment |
| I `command` | Replace the image's default start command | Here it turns on Redis persistence (`--appendonly yes`), so data is written to disk |
| J `volumes` (under a service) | Mount a volume at a path | Without it, Redis data would vanish with the container. `/data` is where Redis writes |
| L `volumes:` (top level) | Declares the volume | Compose must know the volume exists before a service can use it |
| K `healthcheck` | A command Docker runs repeatedly | Answers "is it actually working?", not just "did it start?" |
| F `depends_on` + `service_healthy` | Start order | The API waits until Redis passes its healthcheck, so it never crashes on its first request |

### The two common mix-ups

- **Two different `volumes:`** One sits inside a service (`redisdata:/data` means "mount it here") and one sits at the top level (declares that it exists). You need both.
- **`build` vs `image`.** Your code gets `build`. Someone else's software gets `image`.

## 5. Scaling

```bash
docker compose -f docker-compose.yml -f docker-compose.scale.yml up -d --scale api=3
```

- Three API containers (different IDs) all share one Redis counter.
- This only works because the API is **stateless**: it keeps nothing in itself. Everything that must be remembered lives in Redis.
- A fixed `"8000:8000"` can be used by only one container, so the scale file swaps in a range (`8000-8002:8000`).
- Real systems put a load balancer in front and publish only its port.

## 6. Security habits

- **`.dockerignore`**: keeps `.env`, `.git` and junk out of the image. Anything copied in can be read by anyone who pulls it.
- **Non-root user** (`USER appuser`): if the app is hacked, the attacker isn't root.

## Example: what happens when you run `docker compose up`

```
1. Compose creates a private network      hello_compose_default
2. Compose creates the volume              redisdata
3. Starts redis, mounting redisdata at /data
4. Runs the healthcheck: redis-cli ping ... PONG ... "healthy"
5. Only now starts api (depends_on: service_healthy)
6. api reads REDIS_HOST=redis, connects to the host "redis" over the network
7. You open localhost:8000 -> port mapping -> api container
```

Then `down` and `up` again: new containers, but Redis finds its old data in `redisdata`, so the counter continues.

## Rules to remember

1. Containers are disposable. Data is not. Put data in volumes.
2. Call other containers by name, not by IP.
3. `-p` is for the outside world. Containers on the same network don't need it.
4. "Started" is not "ready". Use healthchecks.
5. Keep apps stateless, so you can replace and multiply them freely.
6. `down` is safe. `down -v` deletes data.